# "THE PRICE IS RIGHT" — Day 2

## Week 6 capstone: predict a product's price from its description

### Order of play
- DAY 1: Data Curation
- **DAY 2: Data Pre-processing** ← *we are here*
- DAY 3: Evaluation, Baselines, Traditional ML
- DAY 4: Deep Learning and LLMs
- DAY 5: Fine-tuning a Frontier Model

### Today — Data Pre-processing
On Day 1 we curated a clean dataset of products + prices. The product text is still messy,
though: scraped titles, feature bullets, and a JSON blob of details. Today we **rewrite each
product into a short, standard format** — title / category / brand / one-line description /
one-line details — using an LLM. Tidy, consistent text gives the downstream models (Day 3
onward) a much better signal to learn from.

### 💡 Business value of data pre-processing / re-writing

LLMs have made something *routine* that was nearly impossible a few years ago: reliably
reshaping free-form text into a clean, structured format at scale. This same move — "rewrite
all of my messy records into one consistent shape" — applies to almost any business vertical,
and it's a close cousin of the document-preprocessing we did in Week 5's advanced RAG lecture.

The heavy lifting lives in `pricer/batch.py`. Rather than calling the LLM once per product
(slow and pricier), we use **Groq's Batch API**: bundle 1,000 requests into a file, submit it,
and collect the results within a 24h window at a steep discount.

In [10]:
# imports
import json

from dotenv import load_dotenv
from litellm import completion

from pricer.items import Item
from pricer.batch import Batch

load_dotenv(override=True)

True

## Choose your dataset size

`LITE_MODE = True` uses the **lite** dataset (20,000 training items) — fast and cheap, ideal for
iterating. `LITE_MODE = False` uses the **full** dataset (800,000 training items) — much heavier.

### What today costs
- **$0** — skip the pre-processing entirely and just load an already-pre-processed dataset
  from the Hub on Day 3 (e.g. `ed-donner/items_lite`).
- **under $1** — run the batch pre-processing for the *lite* dataset.
- **~$30** — run it for the *full* dataset.

In [11]:
LITE_MODE = True

### Load the curated (but not-yet-rewritten) dataset

This is the dataset we pushed on Day 1. Set `username` to your own HuggingFace username to use
*your* Day-1 output, or use `"ed-donner"` to grab the official curated dataset without having run
Day 1 yourself.

In [12]:
username = "marcolerma"  # <-- your HuggingFace username, or "ed-donner" for the official copy
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

# Pre-processing treats every split the same, so work over one combined list and re-split at the end.
items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [13]:
# Items don't have ids yet — they're None
items[2].id

In [14]:
# Give every item a stable id == its index, so we can match batch results back to items
for index, item in enumerate(items):
    item.id = index

### The rewrite prompt

We ask the model to answer in a strict five-line format and nothing else (no prose, no part
numbers). This is the same `SYSTEM_PROMPT` baked into `pricer/batch.py` — shown here so you can
see exactly what each product is rewritten into.

In [15]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

### Try it on a single item first

Before submitting thousands of requests, sanity-check the prompt on one product. We use
[litellm](https://docs.litellm.ai/) so the same `completion()` call works across providers —
here a small open model on Groq, then the same thing locally via Ollama for comparison.

In [16]:
# The raw, messy text we start from
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [17]:
# Rewrite it with a small open model on Groq
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost'] * 100:.3f} cents")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


In [18]:
# The same task, but run locally with Ollama (free; requires `ollama pull llama3.2` and a running server)
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost'] * 100:.3f} cents")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



APIConnectionError: litellm.APIConnectionError: OllamaException - [Errno 61] Connection refused

## Building a batch request file by hand

Before using the `Batch` class, let's build one batch file manually to see what's going on. The
Batch API expects a `.jsonl` file with **one request per line**. Each line carries a `custom_id`
(we use the item id) so we can match each response back to the item it came from.

In [19]:
MODEL = "openai/gpt-oss-20b"

In [20]:
def make_jsonl(item):
    """Render one item as a single Batch-API request line (custom_id = item.id)."""
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
        "reasoning_effort": "low",
    }
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [21]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [22]:
# What one request line looks like
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece m

In [23]:
def make_file(start, end, filename):
    """Write items[start:end] to a .jsonl request file, one request per line."""
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [24]:
import os
os.makedirs("jsonl", exist_ok=True)
make_file(0, 1000, "jsonl/0_1000.jsonl")

### Submit the file manually

Talking to the Batch API is provider-specific, so we use the Groq SDK directly (not litellm).
The flow is: **upload the file → create a batch job → poll until it's `completed` → download the
results**.

In [25]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [26]:
# Upload the request file (binary mode — note: no encoding argument on a binary open)
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

NameError: name 'groq' is not defined

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=file_id,
)
response

In [ ]:
# Re-run this to poll: wait for status == "completed" (can take a while)
result = groq.batches.retrieve(response.id)
result

In [ ]:
# Once completed, download the results file
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
# Match each result back to its item by custom_id, and store the rewritten summary
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary

In [ ]:
# Before: the messy original text
print(items[0].full)

In [ ]:
# After: the clean, standard-format rewrite
print(items[0].summary)

## Doing it properly with the `Batch` class

That whole dance — split into groups of 1,000, write each file, upload, submit, poll, download,
apply — is wrapped up in `pricer/batch.py`. `Batch.create` registers the jobs, `Batch.run`
submits them all, and `Batch.fetch` collects whatever is ready (re-run it until everything is in).

### Costs
On Groq this came to **under \$1** for the lite dataset and **~\$30** for the full one. You can
skip all of it and just load an already-pre-processed dataset on Day 3 — see the note up top.

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
# Re-run this until it reports that all batches are finished
Batch.fetch()

In [ ]:
# Sanity check: which items (if any) still have no summary?
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
# Spot-check a summary from deeper in the dataset
print(items[10234].summary)

### Drop the working fields before pushing

`full` (the messy original) and `id` (a within-this-run index) were only needed for
pre-processing — the downstream models learn from `summary`. Clear them so the published dataset
stays lean.

In [ ]:
for item in items:
    item.full = None
    item.id = None

## Push the pre-processed dataset to the Hub

In lite mode we publish just the lite dataset. In full mode we publish the full dataset **and** a
lite slice of it, so you can switch to lite later without re-running anything. Set `username` to
your own HuggingFace username before pushing.

In [ ]:
username = "marcolerma"  # <-- your HuggingFace username
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## And here they are!

The official pre-processed datasets (handy if you skipped the batch run):
- https://huggingface.co/datasets/ed-donner/items_lite
- https://huggingface.co/datasets/ed-donner/items_full

On Day 3 we'll load one of these and start building actual price models.